<img src="images/strathsdr_banner.png" align="left" >

# RFSoC OFDM Transceiver
----

<div class="alert alert-box alert-info">
Please use Jupyter Labs http://board_ip_address/lab for this notebook.
</div>

This notebook demonstrates the implementation of an Orthogonal Frequency Division Multiplexing (OFDM) transceiver on RFSoC. PYNQ is used to control the underlying modulation scheme of the OFDM sub-carriers and for visualisation of data at various stages in the transmit/receive chain, such as the received constellations.       

## Aims 
* To demonstrate a complete OFDM transceiver.
* Explain the various stages OFDM comprises.
* Provide an interactive and responsive means to inspect the data at each stage.
* Visualise the transmitted and received spectrum.
* Use the OFDM waveform as a radar signal and display a range-Doppler map.

## Table of Contents
* [Introduction](#introduction)
    * [Hardware Setup](#hardware-setup)
    * [Software Setup](#software-setup)
* [Transmit](#transmit)
    * [Symbol Generation](#symbol-gen)
* [Receive](#receive)
    * [Spectrum Analyser](#spectrum-analyser)
    * [Constellation Plot](#constellation-plot)
    * [Range-Doppler Map](#range-doppler)
* [Conclusion](#conclusion)

## References
* [M. Braun, "OFDM Radar Algorithms in Mobile Communication Networks", PhD thesis, Karlsruhe Institute of Technology, 2014](https://publikationen.bibliothek.kit.edu/1000038892)
* [C. Sturm and W. Wiesbeck, "Waveform Design and Signal Processing Aspects for Fusion of Wireless Communications and Radar Sensing", Proceedings of the IEEE, vol. 99, no. 7, 2011](https://doi.org/10.1109/JPROC.2011.2131110)
* [Xilinx, Inc, "USP RF Data Converter: LogiCORE IP Product Guide", PG269, v2.4, November 2020](https://www.xilinx.com/support/documentation/ip_documentation/usp_rf_data_converter/v2_4/pg269-rf-data-converter.pdf)

## Revision History
* **v1.0** | 26/02/2021 | OFDM transceiver notebook
* **v1.1** | 30/03/2022 | OFDM DUC and DDC change
* **v1.2** | 17/05/2023 | General notebook for all boards.
* **v1.3** | 05/11/2026 | Spectrum analyser and range-Doppler map.

----

## Introduction <a class="anchor" id="introduction"></a>

The demonstrator is a complete OFDM transceiver. This notebook will explain each stage of the system with a combination of text, diagrams and live data capture. [Figure 1](#fig-1) below provides an overview of the system.

<a class="anchor" id="fig-1"></a>
<figure>
<img src="./images/system_overview.png" height='75%' width='75%'/>
    <figcaption><b>Figure 1: OFDM Demonstrator System Overview</b></figcaption>
</figure>

The OFDM system starts with generation of random data symbols from 1 of 10 possible modulation schemes (BPSK to 1024 QAM), based on input provided from PYNQ. In accordance with the procedure used in the IEEE 802.11a/g standard, the symbols are grouped into blocks of 48 for mapping to sub-carriers. The OFDM symbol consists of 48 data sub-carriers, 4 pilot sub-carriers and 12 null sub-carriers (including DC). The final OFDM symbol is created by performing a 64 point IFFT and adding a 16 sample Cyclic Prefix (CP). The transmitted signal consists of a continuous stream of OFDM symbols. At the very beginning of the data stream, the L-STF and L-LTF training symbols from the IEEE 802.11a/g standard are transmitted, to aid synchronisation and channel estimation tasks in the receiver. 

In the receiver, timing and frequency synchronisation are performed to acquire symbol timing and correct for any frequency offsets. Once timing is achieved, the FFT is performed to recover the underlying data symbols in each OFDM symbol. The L-LTF symbols are used to estimate the channel frequency response at each sub-carrier position, which subsequently allows the data to be equalised. Finally, the pilot symbols are used to correct for residual phase errors in the phase tracking stage. The recovered symbols are then passed into the PS for visualisation in PYNQ.

### Hardware Setup <a class="anchor" id="hardware-setup"></a>
Your RFSoC development board should be setup in single channel mode with a loopback cable connected between an ADC and DAC.

See the setup instructions [here](01_rfsoc_ofdm_setup.ipynb) for more information.

<div class="alert alert-box alert-danger">
<b>Caution:</b>
    In this demonstration, we generate tones using the RFSoC development board. Your device should be setup in loopback mode. You should understand that the RFSoC platform can also transmit RF signals wirelessly. Remember that unlicensed wireless transmission of RF signals may be illegal in your geographical location. Radio signals may also interfere with nearby devices, such as pacemakers and emergency radio equipment. Note that it is also illegal to intercept and decode particular RF signals. If you are unsure, please seek professional support.
</div>

### Software Setup
The setup for the OFDM demonstration system is nearly complete. The majority of the libraries used by the demonstrator design are contained inside the RFSoC-OFDM software package. We only need to run a few code cells to initialise the environment.

In [ ]:
from rfsoc_ofdm.overlay import Overlay
import ipywidgets as ipw

ofdm_hw = Overlay()

----

## Transmit <a class="anchor" id="transmit"></a>

### Symbol Generation <a class="anchor" id="symbol-gen"></a>
There are a total of 10 modulation schemes available to be transmitted. These are generated on the programmable logic and can be chosen between by updating the *ofdm_tx IP core's* **mod** register with a value from 0-9 over AXI4-Lite. [Figure 2](#fig-2) illustrates the IP core as a simplified block diagram.

<a class="anchor" id="fig-2"></a>
<figure>
<img src="./images/symbol_generation.png" height='45%' width='45%'/>
    <figcaption><b>Figure 2: Symbol generation block diagram.</b></figcaption>
</figure>


This drop down widget sends the value associated with each modulation scheme to the ofdm_tx core. Run the cell to use it.

In [ ]:
ofdm_hw.ofdm_transmitter.modulation_dropdown.get_widget()

The output of the symbol generation block has been tapped off, allowing for the live symbols to be visualised in Jupyter Lab. Run the cell below and hit play on the chart to inspect the symbols generated on the programmable logic.

In [ ]:
ipw.VBox([ofdm_hw.inspectors['transmitter'].time_plot(),
          ofdm_hw.inspectors['transmitter'].plot_control()])

---

## Receive <a class="anchor" id="receive"></a>

### Spectrum Analyser <a class="anchor" id="spectrum-analyser"></a>

The data inspectors can tap the signal at two observation points in both the transmitter and the receiver:

| Observation point | Transmitter | Receiver | Sample rate |
|---|---|---|---|
| **Baseband** (Symbols) | OFDM transmitter output | Input to the OFDM receiver | 20 MSPS |
| **Wideband** (Interpolated / Decimated) | After the 3rd interpolation stage | After the 1st decimation stage | 240 MSPS |

Run the cell below and press play to view the averaged power spectrum of the transmitted and received signals. At the baseband observation point you should see the 52 occupied sub-carriers (about 16.6 MHz wide) with the null DC sub-carrier in the middle. The wideband observation point shows the same signal inside a 240 MHz span. Frequencies are offsets from the centre frequency shown in the title. The power is shown in dBFS per FFT bin, so it drops as the FFT size increases.

In [ ]:
ofdm_hw.spectrum_application()

### Constellation Plot <a class="anchor" id="constellation-plot"></a>

Run the cell below to see the output of the OFDM receiver displayed as a constellation: 

In [ ]:
ipw.VBox([ofdm_hw.inspectors['constellation'].constellation_plot(),
          ofdm_hw.inspectors['constellation'].plot_control()])

Try changing the modulation scheme using the code cell that was ran earlier. You should be able to visualise the modulation schemes in the plot above.

### Range-Doppler Map <a class="anchor" id="range-doppler"></a>

The OFDM waveform can also be used as a radar signal. After cyclic prefix removal and the FFT, sub-carrier $k$ of received OFDM symbol $m$ is a copy of the transmitted modulation symbol $X[m,k]$, weighted by the propagation channel. For paths $p$ with amplitude $a_p$, delay $\tau_p$ and Doppler shift $f_{D,p}$:

$$Y[m,k] = X[m,k] \sum_p a_p \, e^{-j 2\pi k \Delta f \tau_p} \, e^{j 2\pi f_{D,p} m T_\mathrm{sym}} + \mathrm{noise}$$

where $\Delta f = 312.5$ kHz is the sub-carrier spacing and $T_\mathrm{sym} = 4\,\mu$s is the OFDM symbol period (64 + 16 samples at 20 MSPS). Dividing by the transmitted symbols removes the random data, $D[m,k] = Y[m,k] / X[m,k]$. Each path is then a complex sinusoid across sub-carriers, with a frequency set by its delay, and across OFDM symbols, with a frequency set by its Doppler shift. An IFFT across sub-carriers gives the **range** and an FFT across OFDM symbols gives the **Doppler** axis of the map.

The transmitted symbols are random, so the processing captures the transmitter and receiver 20 MSPS observation points at the same time and uses the transmitted samples as the reference. Any modulation scheme works. The two captures start a few microseconds apart, so they are aligned by cross-correlation. Zero range is therefore the **strongest received path**: the loopback cable, or the direct path between the transmit and receive antennas. All other paths are shown relative to it.

| Parameter (256 OFDM symbols, 600 MHz carrier) | Value |
|---|---|
| Range bin, $c / (2 f_s)$ | 7.49 m |
| Range resolution (16.6 MHz occupied bandwidth) | about 9 m |
| Range covered by the cyclic prefix (16 samples less a 4 sample guard) | about 90 m |
| Unambiguous Doppler, $\pm 1 / (2 T_\mathrm{sym})$ | $\pm$125 kHz |
| Doppler resolution, $1 / (M T_\mathrm{sym})$ | 977 Hz (244 Hz with 1024 symbols) |
| Velocity resolution, $c / (2 f_c M T_\mathrm{sym})$ | 244 m/s (61 m/s with 1024 symbols) |

Run the cell below and press play. With a loopback cable there is a single peak at 0 m and 0 Hz. The controls are:

* **OFDM Symbols**: number of OFDM symbols per map. More symbols give a finer Doppler resolution and a lower noise floor.
* **Range Zero-Padding**: interpolation of the range axis.
* **Window**: window applied across sub-carriers and OFDM symbols to reduce sidelobes.
* **Estimator**: *Reciprocal* divides by the transmitted symbols (low range sidelobes). *Matched* multiplies by their conjugate (a matched filter; high order QAM adds a data-dependent noise floor).
* **Remove static clutter**: subtracts the mean over OFDM symbols, which removes everything that does not move, including the direct path.
* **DAC NCO Offset**: shifts the DAC fine mixer to emulate a Doppler shift. See below.

In [ ]:
ofdm_hw.range_doppler_application()

#### Emulating a Moving Target

A loopback cable cannot move, but a Doppler shift can be emulated by offsetting the DAC's numerically controlled oscillator (NCO). The whole received signal is then shifted in frequency. The transmitter reference is captured before the DAC, so the shift appears on the Doppler axis of the map. Drag the **DAC NCO Offset** slider above, or run the cell below, and the peak moves to the chosen Doppler frequency. The hardware OFDM receiver sees the same frequency offset, so the constellation may rotate until the offset is set back to zero.

In [ ]:
ofdm_hw.range_doppler.set_nco_offset(10e3)   # +10 kHz
result = ofdm_hw.range_doppler.measure()
print(result)
ofdm_hw.range_doppler.set_nco_offset(0)      # restore the carrier frequency

#### Working with the Data

The captures and the processed map are available as numpy arrays. `capture()` returns the 20 MSPS transmit and receive samples. `measure()` returns a `RangeDopplerResult` with the map in dB (`rd_map_db`), its axes (`range_m`, `doppler_hz`, `velocity_mps`) and the channel estimate `channel` ($D[m,k]$). The processing parameters are attributes of `ofdm_hw.range_doppler.radar`.

In [ ]:
import numpy as np

tx, rx = ofdm_hw.range_doppler.capture()
result = ofdm_hw.range_doppler.process(tx, rx)
np.savez('ofdm_capture.npz', tx=tx, rx=rx)
print('Map shape (Doppler x range):', result.rd_map_db.shape)
print('Peak range: {:.1f} m, peak Doppler: {:.1f} Hz'.format(result.peak_range_m, result.peak_doppler_hz))

#### Simulated Targets

To see what moving targets look like, the cell below processes a synthetic capture with the same OFDM numerology: a direct path plus two targets at different ranges and velocities. It uses the same processing and plot as the live map above.

In [ ]:
from rfsoc_ofdm.radar import OFDMRadar, simulate_capture
from rfsoc_ofdm.sdr_plots import RangeDopplerPlot

sim_radar = OFDMRadar(max_symbols=256, remove_static=True)
targets = [(0.0, 0.0, 0.0),           # (range m, velocity m/s, power dB): direct path
           (45.0, 7000.0, -10.0),     # target 1
           (75.0, -3000.0, -15.0)]    # target 2
tx_sim, rx_sim = simulate_capture(sim_radar.samples_required(256), targets=targets, snr_db=25)
sim_result = sim_radar.process(tx_sim, rx_sim)

sim_plot = RangeDopplerPlot()
sim_plot.update_data(sim_result, min_range=-30, max_range=150, max_doppler=60e3,
                     units='velocity', title='Simulated Range-Doppler Map')
sim_plot.get_widget()

<div class="alert alert-box alert-warning">
<b>Note:</b> At a 600 MHz carrier, the velocity resolution of a single capture is tens to hundreds of m/s, so the live map is most useful for resolving propagation paths in range and for emulated Doppler shifts. Using antennas instead of a loopback cable means radiating RF energy. Only do this where you are licensed to transmit at the chosen frequency and power.
</div>

## Conclusion <a class="anchor" id="conclusion"></a>
This notebook has demonstrated a live OFDM transceiver operating on an RFSoC. It has been shown how PYNQ can be used to interact with various parts of the hardware design, offering control of the system and visualisation of data. 

* The various components comprising the OFDM transmitter and receiver have been introduced at a high level. 
    * Modulation symbols were inspected.
    * The transmitted and received power spectra were plotted.
    * The received and sychronised constellation were plotted.
    * The OFDM signal was used as a radar waveform to compute a range-Doppler map.
* Interacted with a real-time RF system.
    * Changed modulation scheme.
    * Emulated a Doppler shift with the DAC NCO.

----

⬅️ [Previous Notebook](01_rfsoc_ofdm_setup.ipynb) | [Next Notebook](03_voila_rfsoc_ofdm_demonstrator.ipynb) 🚀

----
----